In [9]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

conn = sqlite3.connect(":memory:")

In [10]:
def q(sql: str, params=None) -> pd.DataFrame:
    return pd.read_sql_query(sql, conn, params=params or {})

In [11]:
## Crear BD y poblarla
conn.executescript("""DROP TABLE IF EXISTS customers;DROP TABLE IF EXISTS products;DROP TABLE IF EXISTS orders;DROP TABLE IF EXISTS order_items;DROP TABLE IF EXISTS payments;CREATE TABLE customers(id_customer INTEGER PRIMARY KEY,name TEXT NOT NULL,region TEXT NOT NULL,signup_date TEXT NOT NULL);CREATE TABLE products(id_product INTEGER PRIMARY KEY,category TEXT NOT NULL,price REAL NOT NULL,active INTEGER NOT NULL);CREATE TABLE orders(id_order INTEGER PRIMARY KEY,id_customer INTEGER NOT NULL,order_date TEXT NOT NULL,status TEXT NOT NULL,FOREIGN KEY(id_customer) REFERENCES customers(id_customer));CREATE TABLE order_items(id_order INTEGER NOT NULL,id_product INTEGER NOT NULL,qty INTEGER NOT NULL,unit_price REAL NOT NULL,FOREIGN KEY(id_order) REFERENCES orders(id_order),FOREIGN KEY(id_product) REFERENCES products(id_product));CREATE TABLE payments(id_payment INTEGER PRIMARY KEY,id_order INTEGER NOT NULL,payment_date TEXT NOT NULL,amount REAL NOT NULL,method TEXT NOT NULL,FOREIGN KEY(id_order) REFERENCES orders(id_order));""")

rng=np.random.default_rng(42)
regions=["Catalunya","Madrid","Andalucía","Valencia","Norte"]
categories=["Electrónica","Hogar","Moda","Deporte","Juguetes"]
methods=["Card","PayPal","Bizum","BankTransfer"]

# customers
n_customers=400
start=datetime(2022,1,1)
customers=[]
for i in range(1,n_customers+1):
    dt=start+timedelta(days=int(rng.integers(0,1100)))
    customers.append((i,f"Customer_{i}",rng.choice(regions),dt.date().isoformat()))
conn.executemany("INSERT INTO customers VALUES (?,?,?,?)",customers)

# products
n_products=120
products=[]
for i in range(1,n_products+1):
    cat=rng.choice(categories)
    price=float(np.round(max(3.0,rng.normal(35,25)),2))
    active=int(rng.random()<0.92)
    products.append((i,cat,price,active))
conn.executemany("INSERT INTO products VALUES (?,?,?,?)",products)

# orders
orders=[]
order_items=[]
payments=[]
id_order=1
id_payment=1

for id_customer in range(1,n_customers+1):
    k=int(rng.integers(0,6))
    for _ in range(k):
        od=start+timedelta(days=int(rng.integers(0,1100)))
        status=rng.choice(["CREATED","PAID","SHIPPED","CANCELLED"],p=[0.10,0.45,0.35,0.10])
        orders.append((id_order,id_customer,od.date().isoformat(),status))
        n_items=int(rng.integers(1,6))
        chosen=rng.choice(np.arange(1,n_products+1),size=n_items,replace=True)
        total=0.0
        for pid in chosen:
            qty=int(rng.integers(1,4))
            unit=float(q("SELECT price FROM products WHERE id_product=:p",{"p":int(pid)}).iloc[0,0])
            order_items.append((id_order,int(pid),qty,unit))
            total+=qty*unit
        if status in ("PAID","SHIPPED"):
            pdte=od+timedelta(days=int(rng.integers(0,10)))
            amt=float(np.round(total+rng.normal(0,1.5),2))
            payments.append((id_payment,id_order,pdte.date().isoformat(),max(0.0,amt),rng.choice(methods)))
            id_payment+=1
        id_order+=1
        
conn.executemany("INSERT INTO orders VALUES (?,?,?,?)",orders)
conn.executemany("INSERT INTO order_items VALUES (?,?,?,?)",order_items)
conn.executemany("INSERT INTO payments VALUES (?,?,?,?,?)",payments)
conn.commit()

In [12]:
# 1) Descubrir la BBDD
print(q("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"))
print(q("PRAGMA table_info(customers);"))
print(q("SELECT * FROM customers LIMIT 5;"))

          name
0    customers
1  order_items
2       orders
3     payments
4     products
   cid         name     type  notnull dflt_value  pk
0    0  id_customer  INTEGER        0       None   1
1    1         name     TEXT        1       None   0
2    2       region     TEXT        1       None   0
3    3  signup_date     TEXT        1       None   0
   id_customer        name     region signup_date
0            1  Customer_1   Valencia  2022-04-09
1            2  Customer_2  Andalucía  2023-12-22
2            3  Customer_3      Norte  2023-04-22
3            4  Customer_4   Valencia  2022-04-05
4            5  Customer_5  Catalunya  2022-08-10


In [13]:
# 2) Preguntas de negocio típicas
# 2.1 Clientes por región
print(q("SELECT region,COUNT(*) AS num_customers FROM customers GROUP BY region ORDER BY num_customers DESC;"))

      region  num_customers
0  Catalunya             88
1  Andalucía             86
2   Valencia             81
3      Norte             75
4     Madrid             70


In [14]:
# 2.2 Pedidos por estado
print(q("SELECT status,COUNT(*) AS num_orders FROM orders GROUP BY status ORDER BY num_orders DESC;"))

      status  num_orders
0       PAID         451
1    SHIPPED         358
2    CREATED         118
3  CANCELLED         103


In [15]:
# 2.3 Ingresos por método de pago
print(q("SELECT method,ROUND(SUM(amount),2) AS revenue FROM payments GROUP BY method ORDER BY revenue DESC;"))

         method   revenue
0        PayPal  48304.76
1  BankTransfer  46082.68
2          Card  42305.33
3         Bizum  41031.06


In [16]:
# 2.4 Top clientes por gasto total (LEFT JOIN para incluir clientes sin compras)
print(q("""SELECT c.id_customer,c.name,ROUND(COALESCE(SUM(p.amount),0),2) AS total_spend FROM customers c LEFT JOIN orders o ON o.id_customer=c.id_customer LEFT JOIN payments p ON p.id_order=o.id_order GROUP BY c.id_customer,c.name ORDER BY total_spend DESC LIMIT 15;"""))

    id_customer          name  total_spend
0           187  Customer_187      1823.42
1           204  Customer_204      1730.69
2           361  Customer_361      1721.57
3           175  Customer_175      1527.95
4           260  Customer_260      1429.44
5           217  Customer_217      1381.65
6           184  Customer_184      1342.88
7            26   Customer_26      1338.11
8           131  Customer_131      1280.49
9           228  Customer_228      1272.90
10          348  Customer_348      1264.78
11          210  Customer_210      1250.01
12           53   Customer_53      1235.35
13          294  Customer_294      1214.50
14          172  Customer_172      1213.66


In [17]:
# 2.5 Clientes con al menos X pedidos (HAVING)
X=3
print(q("""SELECT c.id_customer,c.name,COUNT(o.id_order) AS num_orders FROM customers c LEFT JOIN orders o ON o.id_customer=c.id_customer GROUP BY c.id_customer,c.name HAVING COUNT(o.id_order)>=:x ORDER BY num_orders DESC;""",{"x":X}))

     id_customer          name  num_orders
0              8    Customer_8           5
1             20   Customer_20           5
2             22   Customer_22           5
3             35   Customer_35           5
4             36   Customer_36           5
..           ...           ...         ...
201          383  Customer_383           3
202          395  Customer_395           3
203          396  Customer_396           3
204          397  Customer_397           3
205          399  Customer_399           3

[206 rows x 3 columns]


In [18]:
# 2.6 Ingresos por mes
print(q("""SELECT substr(payment_date,1,7) AS yyyy_mm,ROUND(SUM(amount),2) AS revenue FROM payments GROUP BY substr(payment_date,1,7) ORDER BY yyyy_mm;"""))

    yyyy_mm  revenue
0   2022-01  4688.37
1   2022-02  3315.45
2   2022-03  3946.17
3   2022-04  3137.18
4   2022-05  2674.35
5   2022-06  5210.97
6   2022-07  6126.93
7   2022-08  4946.71
8   2022-09  6431.14
9   2022-10  6880.42
10  2022-11  4779.00
11  2022-12  5573.96
12  2023-01  5413.82
13  2023-02  3854.66
14  2023-03  6121.41
15  2023-04  5577.72
16  2023-05  4453.78
17  2023-06  5435.58
18  2023-07  5655.02
19  2023-08  3786.38
20  2023-09  6748.67
21  2023-10  7969.45
22  2023-11  3990.76
23  2023-12  4030.87
24  2024-01  3809.26
25  2024-02  3871.03
26  2024-03  4973.97
27  2024-04  3548.63
28  2024-05  7047.28
29  2024-06  4478.46
30  2024-07  6273.36
31  2024-08  4035.35
32  2024-09  3733.63
33  2024-10  7308.06
34  2024-11  3040.65
35  2024-12  4166.45
36  2025-01   688.93


In [19]:
# 2.7 Top categorías por unidades vendidas
print(q("""SELECT pr.category,SUM(oi.qty) AS units FROM order_items oi JOIN products pr ON pr.id_product=oi.id_product JOIN orders o ON o.id_order=oi.id_order WHERE o.status IN ('PAID','SHIPPED') GROUP BY pr.category ORDER BY units DESC;"""))

      category  units
0      Deporte   1099
1     Juguetes   1098
2         Moda   1053
3  Electrónica    774
4        Hogar    759


In [20]:
# 2.8 Ticket medio (AOV) sobre pedidos pagados
print(q("""SELECT ROUND(AVG(p.amount),2) AS avg_order_value FROM payments p;"""))

   avg_order_value
0           219.68


In [21]:
# 3) Calidad de datos
# 3.1 Duplicados
print(q("SELECT COUNT(*) AS total,COUNT(DISTINCT id_customer) AS distinct_customers FROM customers;"))

   total  distinct_customers
0    400                 400


In [22]:
# 3.2 NULLs / valores sospechosos
print(q("SELECT SUM(CASE WHEN price IS NULL OR price<=0 THEN 1 ELSE 0 END) AS bad_prices,COUNT(*) AS total FROM products;"))

   bad_prices  total
0           0    120


In [23]:
# 3.3 Órdenes huérfanas (integridad referencial)
print(q("""SELECT COUNT(*) AS orphan_orders FROM orders o LEFT JOIN customers c ON c.id_customer=o.id_customer WHERE c.id_customer IS NULL;"""))

   orphan_orders
0              0


In [24]:
# 4) Índices (conceptual)
conn.execute("CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(id_customer)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_payments_order ON payments(id_order)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(id_order)")
conn.execute("CREATE INDEX IF NOT EXISTS idx_items_product ON order_items(id_product)")
conn.commit()
print("Indexes created")

Indexes created
